# m09 lab · Propagation: cable, monodomain, bidomain, anisotropy

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Couple the m08 cells with conductances and the action potential travels. You start with the passive cable, measure
conduction velocity on an excitable cable, reduce the bidomain model to the monodomain one, spread a wave from a
point in anisotropic tissue, and see a curved front slow down.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The passive cable and its length constant | eq-m09-cable |
| B | Velocity on an excitable cable: square-root scaling and grid error | eq-m09-cv |
| C | From bidomain to monodomain; anisotropy ratios from measured resistances | eq-m09-bidomain, eq-m09-monodomain |
| D | Spread from a point in anisotropic tissue | eq-m09-anisotropy |
| E | Curved fronts and the stimulus that fails | eq-m09-curvature |

Membrane kinetics are Mitchell–Schaeffer (m08). Distances in mm, times in ms, velocities in mm/ms (= m/s). Each part
ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import propagation as G

## Part A · The passive cable

Below threshold a fiber is a leaky cable (eq-m09-cable). Hold one end at a voltage and it decays as
$e^{-x/\lambda}$ with $\lambda = \sqrt{r_m/(r_i + r_e)}$. Check the finite-difference solution against it.

In [ ]:
lam = 2.0                                                 # mm, illustrative
errs = []
for dx in (0.2, 0.1, 0.05):
    n = int(round(40 / dx))
    xs = (np.arange(n) + 1) * dx
    errs.append(np.max(np.abs(G.cable_steady_fd(n, dx, lam) - G.cable_steady(xs, 1.0, lam))))
    print(f"dx = {dx} mm: largest error {errs[-1]:.1e}")
xs = np.linspace(0, 10, 200)
plt.plot(xs, G.cable_steady(xs, 1.0, lam)); plt.axvline(lam, color="gray", ls="--")
plt.xlabel("distance from the held end (mm)"); plt.ylabel("V / V0"); plt.show()
print(f"at one length constant: {G.cable_steady(lam, 1.0, lam):.3f} of the held voltage")

In [ ]:
# Checks
assert errs[0] / errs[1] > 3.5 and errs[1] / errs[2] > 3.5   # second order
assert abs(G.cable_steady(lam, 1.0, lam) - np.exp(-1)) < 1e-12

## Part B · Velocity on an excitable cable

In the monodomain equation conductivity enters only through $D = \sigma/(\chi C_m)$. Stretching space by $\sqrt k$
turns $D$ into $kD$, so velocity scales as $\sqrt D$ (eq-m09-cv). Measure it, tune $D$ for 1.0 m/s, the velocity
reported for normal-voltage left atrium (Heil et al. 2023), and see how a coarse grid slows the computed wave.

In [ ]:
Ds = [0.1, 0.2, 0.4, 0.8]
cvs = [G.cable_cv(D) for D in Ds]
for D, c in zip(Ds, cvs):
    print(f"D = {D:.1f} mm^2/ms: velocity {c:.3f} m/s, velocity / sqrt(D) = {c / np.sqrt(D):.3f}")
D1 = G.d_for_cv(1.0)
print(f"D for 1.0 m/s: {D1:.3f} mm^2/ms")
grid = {dx: G.cable_cv(0.1, dx=dx) for dx in (0.05, 0.1, 0.2, 0.4)}
for dx, c in grid.items():
    print(f"grid {dx} mm: velocity {c:.4f} m/s")

In [ ]:
# Checks
ratios = np.array(cvs) / np.sqrt(Ds)
assert np.ptp(ratios) / ratios.mean() < 0.01
assert abs(G.cable_cv(D1) - 1.0) < 0.01
assert grid[0.4] < grid[0.05]                               # coarse grids slow the wave

## Part C · From bidomain to monodomain

The bidomain model keeps intracellular and extracellular conductivities separate (eq-m09-bidomain). In 1D, with no
net current along the fiber, the extracellular potential mirrors the transmembrane one,
$\varphi_e = -\sigma_i V/(\sigma_i + \sigma_e)$, and the bidomain collapses to a monodomain model with the harmonic
combination $\sigma_i\sigma_e/(\sigma_i + \sigma_e)$ (eq-m09-monodomain). In 2D or 3D that works only if both domains
have the same anisotropy ratio. Clerc's measurements in calf trabeculae say they do not (1976).

In [ ]:
si_L, si_T, se_L, se_T = G.clerc_conductivities()
print(f"relative conductivities: intracellular {si_L:.3f} (L) / {si_T:.3f} (T); extracellular {se_L:.3f} (L) / {se_T:.3f} (T)")
print(f"anisotropy ratios: intracellular {si_L / si_T:.1f}, extracellular {se_L / se_T:.1f}")
pred = G.cv_ratio_from_conductivities(si_L, si_T, se_L, se_T)
print(f"predicted velocity ratio sqrt(sigma_m,L / sigma_m,T) = {pred:.2f}; measured {G.CLERC['cv_ratio']}")
V = np.tanh(np.linspace(-3, 3, 61))                         # any 1D voltage profile
phi_e = -si_L / (si_L + se_L) * V
net = (si_L + se_L) * np.gradient(phi_e) + si_L * np.gradient(V)
print(f"largest net axial current with phi_e = -sigma_i V / (sigma_i + sigma_e): {np.max(np.abs(net)):.1e}")

In [ ]:
# Checks
assert abs(pred - 3.0) / 3.0 < 0.07
assert np.max(np.abs(net)) < 1e-12
assert abs((si_L / si_T) - 9.4) < 1e-9 and abs((se_L / se_T) - 2.7) < 1e-9

Potse and colleagues compared full monodomain and bidomain models of the human heart and found propagation only 2%
faster in the bidomain model, with visually indistinguishable electrograms, in the absence of applied currents (2006).
Stimulation and defibrillation are where the bidomain model matters.

## Part D · Spread from a point in anisotropic tissue

With fibers along $x$, $D_L$ and $D_T$ set the two velocities and the far-field isochrones are ellipses
(eq-m09-anisotropy). Simulate a sheet with $D_T = D_L/9$, a velocity ratio of 3, and measure the speed along each
axis at increasing distances. About 5 s.

In [ ]:
n, dx = 201, 0.2
act = G.sheet_activation(n, dx, 0.01, 60.0, D1, D1 / 9)
x = (np.arange(n) - 100) * dx
speeds = {}
for lo, hi in ((5, 10), (10, 15), (14, 19)):
    sel = (x > lo) & (x < hi)
    speeds[lo] = (1 / np.polyfit(x[sel], act[100, sel], 1)[0], 1 / np.polyfit(x[sel], act[sel, 100], 1)[0])
    print(f"{lo}-{hi} mm from the stimulus: along fibers {speeds[lo][0]:.3f}, across {speeds[lo][1]:.3f} m/s, "
          f"ratio {speeds[lo][0] / speeds[lo][1]:.2f}")
plt.figure(figsize=(4.5, 4))
plt.contour(x, x, act, levels=np.arange(5, 60, 5)); plt.gca().set_aspect("equal")
plt.xlabel("along fibers (mm)"); plt.ylabel("across fibers (mm)"); plt.show()
for deg in (0, 30, 60, 90):
    print(f"mapping line at {deg:2d} degrees to the fibers: apparent speed {G.ray_speed(np.radians(deg), 1.0, 1 / 3):.2f} m/s")

In [ ]:
# Checks
assert abs(speeds[14][0] / speeds[14][1] - 3.0) < 0.1
assert speeds[5][0] < speeds[10][0] < speeds[14][0] < 1.0   # the front flattens and speeds up toward 1.0

## Part E · Curved fronts

A convex front feeds more tissue ahead than behind, so it slows: $c \approx c_0 - D\kappa$, with $\kappa = 1/r$ for an
expanding circle (eq-m09-curvature). Simulate a circular wave by the radially symmetric equation, then shrink the
stimulated disk until the wave fails to launch.

In [ ]:
D = 0.1
c0 = G.cable_cv(D)
r, act_r = G.radial_wave(D, dr=0.05, dt=0.002, radius=10.0, t_end=60.0, r_stim=1.0)
sp_r = G.front_speed(r, act_r)
for R in (2.0, 3.0, 5.0, 8.0):
    i = np.argmin(np.abs(r - R))
    print(f"r = {R} mm: front speed {sp_r[i]:.3f}, c0 - D/r = {c0 - D / R:.3f} m/s")
launch = {}
for rs in (0.1, 0.15, 0.2, 0.3, 0.5):
    rr, aa = G.radial_wave(D, dr=0.02, dt=0.0005, radius=3.0, t_end=15.0, r_stim=rs)
    launch[rs] = bool(np.isfinite(aa[-1]))
    print(f"stimulated disk of radius {rs} mm: wave {'launched' if launch[rs] else 'failed'}")
print(f"critical radius estimate D / c0 = {D / c0:.2f} mm")

In [ ]:
# Checks
for R in (3.0, 5.0, 8.0):
    i = np.argmin(np.abs(r - R))
    assert abs(sp_r[i] / (c0 - D / R) - 1) < 0.02
assert launch[0.5] and not launch[0.1]

## Where to go next

- Repeat part D with the two velocities from Roberts and colleagues' canine epicardium (a ratio of 2.4).
- Put a narrow isthmus between two wide regions: where does the wave slow or block as it emerges? m10 uses this for
  reentry.
- Rotate the fibers in part D and measure velocity along a fixed mapping line, as m25 does on real maps.